# Los mecanismos de atención — el cuaderno que los corre

Este cuaderno **corre** la comparación de la Sección 4 y nada más: los cinco
mecanismos que `wiring.MECHANISMS` declara, sobre el método completo (`G`,
nunca un id declarado — `wiring.MechanismArm` hereda todo lo demás de `G` por
herencia y nunca por copia), en las dos condiciones que la sección lee: limpio
y contaminado. No arma ninguna tabla ni conclusión — eso lo hace
`Results.ipynb`, que lee el registro que esto deja
(`tables.MECHANISM_RECORD`) y lo presenta con
`tables.render_mechanisms`/`conclusion_mechanisms`.

Existía la mitad que dibuja y no la que corre: el paso `mechanisms`
(`steps.mecanismos_de_atencion`) llamaba a
`harness.run_mechanism_sweep_shard()` desde la biblioteca, sin cuaderno
propio, así que el ensayo lo recorría probando la biblioteca y dejando sin
ejercitar el artefacto.

La biblioteca no se toca y este cuaderno no reimplementa el barrido: la celda
de la corrida llama a `harness.run_mechanism_sweep_shard()`, que es la MISMA
entrada que un `run-config.json` remoto nombra
(`{module: "MIL_CREDA_Benchmark.harness", function:
"run_mechanism_sweep_shard"}`).


> **Un cuaderno aparte del de la campaña, y no una sección suya.**
>
> Los dos corren a escala completa y los dos consumen el mismo registro de
> techos, así que juntarlos era posible. Lo que costaría está en la aritmética
> de abajo: este barrido son cinco mecanismos sobre una configuración, y la
> campaña es la rejilla entera. Fundirlos ataría para siempre el barato al
> caro — re-correr la Sección 4 pediría volver a pagar la campaña — y es
> exactamente la separación que la búsqueda y su corrida ya tienen.
>
> **Y no tiene modo de ensayo tampoco.** `run_mechanism_sweep_shard` fija
> `config.FULL_EPOCHS`/`config.FULL_SEEDS` adentro y no toma ningún dial,
> la misma decisión que `run_campaign_shard` y por la misma razón: una
> comparación medida a tres épocas es otro experimento, no uno más barato.


In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
from MIL_CREDA_Benchmark import config, harness, tables, wiring

# La escala de esta corrida, leída de las dos constantes que
# `run_mechanism_sweep_shard` usa de verdad --- `FULL_EPOCHS` y `FULL_SEEDS`
# --- y no de `config.EPOCHS`/`config.SEEDS`, que son las del recorrido
# configurado. Las dos pueden diferir, y un encabezado que dijera la del
# recorrido sobre una corrida completa describiría una corrida que no pasó.
EPOCAS = config.FULL_EPOCHS
SEMILLAS = list(config.FULL_SEEDS)

print("escala: completa")
print(f"  {EPOCAS} épocas, {len(SEMILLAS)} semillas")
print(f"  (el recorrido configurado está en {config.EPOCHS} épocas y "
      f"{len(config.SEEDS)} semillas)")

# El destino, nombrado por la constante que lo declara y no compuesto acá.
# Es la raíz que el paso `mechanisms` declara en `produces`, y es la única que
# escribe. No lleva coordenada de escala y eso es una afirmación, no un
# descuido: `config.DESTINOS_SIN_COORDENADA` ya la declara con su razón ---
# `run_mechanism_sweep` multiplexa limpio y contaminado ADENTRO del JSON
# (`clean`/`noisy`), nunca por directorio ni por nombre de archivo, y la
# Sección 4 no declara un `Reduction` de ensayo propio.
print("escribe en:", tables.MECHANISM_RECORD)

## Lo que consume, y de qué árbol

Como la campaña, esta comparación no busca su propio techo: lo consume.
`run_mechanism_sweep_shard` llama a `harness.with_ceilings_in_force`, que lee
el registro de la corrida COMPLETA y se niega si no está.

Se imprime de qué archivo sale y qué dice de sí mismo antes de gastar nada.


In [ ]:
# `pilot=False` escrito y no omitido, por la misma razón que en la campaña: la
# omisión de `ceilings_record_for` no tiene un "el que rige" al que
# replegarse, y un registro de ensayo no puede gobernar una corrida completa.
REGISTRO_TECHOS = config.ceilings_record_for(pilot=False)
print("techos leídos de:", REGISTRO_TECHOS)
print(harness.search_source_note(pilot=False))

## Lo que cuesta

Cinco mecanismos, sobre cada transferencia del veredicto, por semilla, y dos
veces: una limpia y una contaminada a `config.NOISE_REPORTED`. Las dos
condiciones viven adentro de UNA llamada, a propósito — un envío solo deja el
registro completo en vez de dos que alguien tiene que acordarse de mandar.


In [ ]:
MECANISMOS = list(wiring.MECHANISMS)
TRANSFERENCIAS = list(config.VERDICT_TRANSFERS)
CONDICIONES = (0.0, config.NOISE_REPORTED)
CORRIDAS = (len(MECANISMOS) * len(TRANSFERENCIAS) * len(SEMILLAS)
            * len(CONDICIONES))

print(f"mecanismos: {len(MECANISMOS)} — {', '.join(MECANISMOS)}")
print(f"transferencias: {len(TRANSFERENCIAS)}")
print(f"semillas: {len(SEMILLAS)}")
print("condiciones:", ", ".join(f"rho={tasa:g}" for tasa in CONDICIONES))
print(f"corridas de entrenamiento: {CORRIDAS}, de {EPOCAS} épocas cada una")

## La corrida

Una sola llamada, que corre las dos condiciones en orden: primero la limpia
(`rho=0`), después la contaminada (`config.NOISE_REPORTED`). El registro queda
con las dos claves, y una corrida parcial — la limpia terminada y la
contaminada no — es representable en disco, que es lo que `run_mechanism_sweep`
escribiendo por clave compra.


In [ ]:
registro = harness.run_mechanism_sweep_shard()

print()
print("mecanismos en el registro:", ", ".join(registro.get("mechanisms", [])))
print("corridas limpias:", len(registro.get("clean", [])))
print("corridas contaminadas:", len(registro.get("noisy", [])))
print("la tabla y la conclusión de la Sección 4 las arma Results.ipynb")

In [ ]:
# El sello: contra qué código corrió este barrido. Sin él, un registro viejo y
# uno recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())